In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv(r"E:\nachiket\RP\Dataset-SA-Augmented.csv")
df = df[["Review", "Sentiment"]].dropna()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
hinglish_map = {
    "acha": "good", "accha": "good", "mast": "great", "badiya": "great",
    "bakwas": "bad", "bekar": "bad", "bakwass": "bad",
    "bhut": "very", "bahut": "very", "kamal": "excellent",
    "ghatiya": "bad", "sahi": "nice", "faltu": "useless",
    "jhakas": "awesome", "jabardast": "awesome"
}

In [ ]:
def clean_text(t):
    t = t.lower()
    for hi, en in hinglish_map.items():
        t = re.sub(r"\b" + hi + r"\b", en, t)
    t = re.sub(r"http\S+|www\S+", "", t)                  
    t = re.sub(r"[^\w\s]", " ", t)                      
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)                
    t = re.sub(r"\s+", " ", t).strip()                   
    return t


df["Cleaned"] = df["Review"].apply(clean_text)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["Cleaned"], df["Sentiment"],
    test_size=0.2, stratify=df["Sentiment"], random_state=42
)

In [ ]:
tfidf_word = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=80000,
    stop_words="english",
    sublinear_tf=True
)

tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=50000,
    sublinear_tf=True
)

vectorizer = FeatureUnion([
    ("word", tfidf_word),
    ("char", tfidf_char)
])

In [ ]:
svm = LinearSVC(C=1.5)

In [ ]:
model = Pipeline([
    ("vectorizer", vectorizer),
    ("clf", svm)
])

In [ ]:
print("Running 5-Fold Cross-Validation")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("CV Accuracy Scores:", cv_scores)
print("Mean CV Accuracy:", np.mean(cv_scores))

In [ ]:
print("Training final model on full training set")
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Final Test Accuracy:", accuracy_score(y_test, preds))
print("Classification Report:\n", classification_report(y_test, preds))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, preds, labels=model.named_steps["clf"].classes_)
labels = model.named_steps["clf"].classes_

plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — TF-IDF + Linear SVC")
plt.show()


In [ ]:
print("Original full dataset:", df["Sentiment"].value_counts())
print("Training set:", pd.Series(y_train).value_counts())
print("Test set:", pd.Series(y_test).value_counts())


In [ ]:
!pip install lime

In [ ]:
import importlib
import lime
importlib.reload(lime)


In [ ]:
import numpy as np
from lime.lime_text import LimeTextExplainer
import IPython.display as disp


In [ ]:
class_names = list(model.named_steps["clf"].classes_)
explainer = LimeTextExplainer(class_names=class_names)

In [ ]:
def svm_predict_proba(texts):
    vect = model.named_steps["vectorizer"].transform(texts)
    clf = model.named_steps["clf"]

    # LinearSVC gives decision_function values
    df_scores = clf.decision_function(vect)

    # If binary, convert shape (n,) → (n,2)
    if df_scores.ndim == 1:
        df_scores = np.vstack([-df_scores, df_scores]).T

    # Softmax for pseudo probabilities
    exp = np.exp(df_scores - np.max(df_scores, axis=1, keepdims=True))
    probs = exp / np.sum(exp, axis=1, keepdims=True)

    return probs

In [ ]:
def explain_review(text, num_features=10):
    print("Text:", text, "\n")
    probs = svm_predict_proba([text])[0]
    pred = class_names[np.argmax(probs)]

    print("Prediction:", pred)
    print("Probabilities:", {c: float(p) for c, p in zip(class_names, probs)})

    exp = explainer.explain_instance(
        text,
        svm_predict_proba,
        num_features=num_features,
        num_samples=600  
    )

    html = exp.as_html()
    disp.display(disp.HTML(html))

    with open("lime_explanation.html", "w", encoding="utf-8") as f:
        f.write(html)
    print("\nSaved explanation to lime_explanation.html")

    return exp



In [ ]:
sample_indices = np.random.choice(len(X_test), size=3, replace=False)

for idx in sample_indices:
    text = X_test.iloc[idx]
    true_label = y_test.iloc[idx]

    print("True Label:", true_label)
    explain_review(text)


In [ ]:
explain_review("camera quality is good but battery drains fast")


In [ ]:
explain_review("Battery drains fast.")


In [ ]:
explain_review("Worst product.")


In [ ]:
explain_review("Good product but bad battery life")


In [ ]:
explain_review("Okayish product.")


In [ ]:
explain_review("mid product")


In [ ]:
explain_review("Product could be better")


In [ ]:
explain_review("not too good, not too bad")


In [ ]:
import joblib
joblib.dump(model, "tfidf_logreg_model.pkl")
print("Model saved.")
